# 01 — Tratamento da base: SIH-PB 2025 unificado com nomes de municípios

**Story:** US-01 · **O que este notebook faz:** pega os 12 arquivos mensais de internações
da Paraíba (SIH/DATASUS, ano de 2025) que estão congelados em `data/raw/`, junta tudo
num único dataset, e traduz os códigos de município para nomes legíveis — para que
toda análise seguinte fale "João Pessoa", não "250750".

**Por que isso importa:** as colunas centrais do projeto são:
- `MUNIC_RES` — município onde o paciente **mora** (origem)
- `MUNIC_MOV` — município onde o paciente **se internou** (destino)

É a comparação entre as duas que revela a evasão assistencial (morar num lugar e
se internar em outro). Mas ambas vêm como códigos numéricos do IBGE — ilegíveis.
Este notebook resolve isso e valida que nada se perdeu no caminho.

**Validações que serão impressas ao longo do notebook:**
1. Total de linhas do dataset unificado == soma das linhas dos 12 arquivos mensais.
2. 100% dos códigos de `MUNIC_RES` e `MUNIC_MOV` com nome de município atribuído
   (zero códigos "órfãos" — sem correspondência na tabela do IBGE).

**Saída:** `data/processed/sih_pb_2025_tratado.parquet`

> Este notebook roda 100% offline: os dados do SIH já estão congelados em parquet e a
> tabela de municípios do IBGE já está salva no repositório (baixada uma única vez por
> `src/baixar_municipios_ibge.py`).

## 1. Preparação

Importamos as bibliotecas e definimos os caminhos das pastas. Usamos `pathlib.Path`
em vez de strings de caminho porque ele funciona igual em Windows, Linux e Mac
(barras `/` vs `\` deixam de ser problema). Os caminhos são relativos à pasta
`notebooks/`, onde este arquivo vive.

In [1]:
from pathlib import Path

import pandas as pd

# Pastas do projeto (relativas à pasta notebooks/)
DATA_RAW = Path("..") / "data" / "raw"
DATA_PROCESSED = Path("..") / "data" / "processed"

print(f"pandas {pd.__version__}")
print(f"Pasta de dados brutos existe? {DATA_RAW.exists()}")

pandas 2.3.3
Pasta de dados brutos existe? True


## 2. Carregar os 12 arquivos mensais

Cada arquivo `sih_pb_2025_MM.parquet` contém as internações (AIHs — Autorizações de
Internação Hospitalar) registradas em hospitais da Paraíba naquele mês de 2025.

A estratégia: carregar cada mês num DataFrame separado, **anotar quantas linhas cada
um tem** (vamos precisar disso na validação de soma), e só depois juntar tudo.
Guardar a contagem por mês ANTES de concatenar é o que nos permite provar, no passo
seguinte, que nenhuma linha sumiu ou foi duplicada na junção.

In [2]:
arquivos_mensais = sorted(DATA_RAW.glob("sih_pb_2025_*.parquet"))

assert len(arquivos_mensais) == 12, (
    f"Esperava 12 arquivos mensais, encontrei {len(arquivos_mensais)}. "
    "Confira a pasta data/raw/."
)

dfs_mensais = []       # lista com o DataFrame de cada mês
linhas_por_mes = {}    # dicionário mês -> nº de linhas (para a validação de soma)

for arquivo in arquivos_mensais:
    df_mes = pd.read_parquet(arquivo)
    mes = arquivo.stem.split("_")[-1]  # "sih_pb_2025_01" -> "01"
    linhas_por_mes[mes] = len(df_mes)
    dfs_mensais.append(df_mes)
    print(f"  mês {mes}: {len(df_mes):>7,} internações".replace(",", "."))

soma_dos_meses = sum(linhas_por_mes.values())
print(f"\nSoma das contagens dos 12 meses: {soma_dos_meses:,}".replace(",", "."))

  mês 01:  20.029 internações
  mês 02:  19.487 internações


  mês 03:  21.502 internações
  mês 04:  21.058 internações


  mês 05:  22.266 internações


  mês 06:  22.325 internações
  mês 07:  22.816 internações


  mês 08:  23.200 internações


  mês 09:  22.459 internações
  mês 10:  23.150 internações


  mês 11:  20.585 internações


  mês 12:  19.248 internações

Soma das contagens dos 12 meses: 258.125


## 3. Concatenar em um único dataset

`pd.concat` empilha os 12 DataFrames um embaixo do outro (os 12 têm exatamente as
mesmas colunas, então as linhas apenas se acumulam). O `ignore_index=True` descarta
os índices originais de cada mês e cria uma numeração nova de 0 até o total — sem
isso, teríamos índices repetidos (cada mês recomeça do 0), o que causa bugs sutis
em operações futuras.
> **Nota sobre as colunas:** os meses de janeiro e fevereiro vieram do DATASUS com 113
> colunas; de março em diante os arquivos ganharam a coluna `FONTE_ORC` (fonte
> orçamentária), totalizando 114. O `pd.concat` alinha as colunas pelo nome e preenche
> `FONTE_ORC` com vazio (`NaN`) em jan/fev — nenhuma linha é afetada, e a coluna não é
> usada nas análises deste projeto.


In [3]:
sih = pd.concat(dfs_mensais, ignore_index=True)

print(f"Dataset unificado: {len(sih):,} linhas x {sih.shape[1]} colunas".replace(",", "."))

Dataset unificado: 258.125 linhas x 114 colunas


### ✅ Validação 1 — nenhuma linha perdida ou duplicada na junção

O total do dataset unificado tem que ser **exatamente igual** à soma das contagens
dos 12 arquivos de origem. Se der diferente, algo se perdeu (ou duplicou) na
concatenação e nada adiante é confiável. O `assert` interrompe o notebook na hora
se a igualdade falhar — validação não é print decorativo, é trava.

In [4]:
total_unificado = len(sih)

print(f"Total do dataset unificado : {total_unificado:,}".replace(",", "."))
print(f"Soma dos 12 meses          : {soma_dos_meses:,}".replace(",", "."))

assert total_unificado == soma_dos_meses, "FALHOU: total != soma dos meses!"
print("\n✅ VALIDAÇÃO 1 OK — total unificado == soma dos 12 meses")

Total do dataset unificado : 258.125
Soma dos 12 meses          : 258.125

✅ VALIDAÇÃO 1 OK — total unificado == soma dos 12 meses


## 4. Tipar as colunas-chave

Os parquets do SIH guardam quase tudo como texto. Antes de qualquer join ou análise,
garantimos explicitamente o tipo das colunas que este projeto usa como chave:

- **`MUNIC_RES` e `MUNIC_MOV`** → texto (string) de **6 dígitos**. Parece
  contraintuitivo manter código como texto em vez de número, mas é proposital:
  códigos IBGE podem começar com zero em outros estados (ex.: Rondônia, `11...`,
  não, mas SP `35...`; o ponto real é: código é identificador, não quantidade —
  ninguém soma códigos de município). Converter para número arriscaria perder
  zeros à esquerda e quebraria o join com a tabela do IBGE.
- **`ANO_CMPT` e `MES_CMPT`** (ano/mês de competência do registro) → inteiros,
  porque serão usados para ordenar e filtrar por período.

In [5]:
# Códigos de município: string, exatamente 6 dígitos (zfill repõe zeros à esquerda
# caso alguma leitura os tenha descartado)
sih["MUNIC_RES"] = sih["MUNIC_RES"].astype(str).str.zfill(6)
sih["MUNIC_MOV"] = sih["MUNIC_MOV"].astype(str).str.zfill(6)

# Ano e mês de competência: inteiros
sih["ANO_CMPT"] = sih["ANO_CMPT"].astype(int)
sih["MES_CMPT"] = sih["MES_CMPT"].astype(int)

# Conferência rápida: todos os códigos ficaram com 6 caracteres?
print("Tamanhos distintos em MUNIC_RES:", sih["MUNIC_RES"].str.len().unique())
print("Tamanhos distintos em MUNIC_MOV:", sih["MUNIC_MOV"].str.len().unique())
print("Anos presentes:", sorted(sih["ANO_CMPT"].unique()))
print("Meses presentes:", sorted(sih["MES_CMPT"].unique()))

Tamanhos distintos em MUNIC_RES: [6]


Tamanhos distintos em MUNIC_MOV: [6]
Anos presentes: [np.int64(2025)]
Meses presentes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


## 5. A tabela código → nome do IBGE (e a pegadinha dos 6 vs 7 dígitos)

A tabela `data/raw/municipios_ibge.csv` foi baixada da API oficial de Localidades do
IBGE (`https://servicodados.ibge.gov.br/api/v1/localidades/municipios`) pelo script
`src/baixar_municipios_ibge.py` e está versionada no repositório — ou seja, este
notebook **não** acessa a internet.

**A pegadinha:** o código IBGE oficial tem **7 dígitos** (ex.: `2507507` =
João Pessoa), mas o último dígito é apenas um **dígito verificador** — um dígito de
conferência calculado a partir dos outros seis, como o último dígito do CPF. O SIH
grava o código **sem** esse verificador, com **6 dígitos** (`250750`).

Por isso o CSV traz as duas formas: `codigo_ibge7` (oficial) e `codigo_ibge6`
(os 6 primeiros dígitos). **O join usa `codigo_ibge6`.** Isso é seguro porque o
prefixo de 6 dígitos é único no Brasil inteiro — o script de download verifica essa
unicidade com um `assert`, e nós reconferimos aqui.

**Detalhe importante:** a tabela é **nacional** (todos os ~5,5 mil municípios do
Brasil), não só da PB. De propósito: `MUNIC_RES` pode trazer pacientes de outros
estados internados na PB, e queremos dar nome a eles também, não tratá-los como erro.

In [6]:
municipios = pd.read_csv(
    DATA_RAW / "municipios_ibge.csv",
    dtype=str,  # tudo como texto: códigos são identificadores, não números
)

print(f"{len(municipios):,} municípios na tabela do IBGE".replace(",", "."))
print(f"Deles, {(municipios['uf'] == 'PB').sum()} são da Paraíba")

# Reconferindo a premissa que torna o join de 6 dígitos seguro:
assert municipios["codigo_ibge6"].is_unique, "Código de 6 dígitos duplicado!"
print("✅ Código de 6 dígitos é único nacionalmente — join seguro")

municipios.head(3)

5.571 municípios na tabela do IBGE
Deles, 223 são da Paraíba
✅ Código de 6 dígitos é único nacionalmente — join seguro


,codigo_ibge7,codigo_ibge6,nome,uf
0,1100015,110001,Alta Floresta D'Oeste,RO
1,1100023,110002,Ariquemes,RO
2,1100031,110003,Cabixi,RO


## 6. Juntar nomes de origem e de destino

Fazemos **dois** joins com a mesma tabela — um para a origem, outro para o destino —
porque cada linha da base tem dois códigos de município diferentes a traduzir:

| coluna do SIH | vira | significado |
|---|---|---|
| `MUNIC_RES` | `nome_mun_res`, `uf_res` | onde o paciente mora (origem) |
| `MUNIC_MOV` | `nome_mun_mov`, `uf_mov` | onde ele se internou (destino) |

Usamos `merge` com `how="left"`: mantém **todas** as linhas do SIH e, onde o código
não existir na tabela do IBGE, deixa o nome vazio (`NaN`) em vez de descartar a
linha. Isso é essencial para a validação seguinte — um código sem nome deve **aparecer**
como órfão para ser investigado, nunca sumir silenciosamente da base.

In [7]:
linhas_antes_dos_joins = len(sih)

# Join 1 — nome do município de RESIDÊNCIA (origem)
sih = sih.merge(
    municipios[["codigo_ibge6", "nome", "uf"]].rename(
        columns={"nome": "nome_mun_res", "uf": "uf_res"}
    ),
    left_on="MUNIC_RES",
    right_on="codigo_ibge6",
    how="left",
).drop(columns="codigo_ibge6")

# Join 2 — nome do município de INTERNAÇÃO (destino)
sih = sih.merge(
    municipios[["codigo_ibge6", "nome", "uf"]].rename(
        columns={"nome": "nome_mun_mov", "uf": "uf_mov"}
    ),
    left_on="MUNIC_MOV",
    right_on="codigo_ibge6",
    how="left",
).drop(columns="codigo_ibge6")

# Um merge mal feito (chave duplicada do outro lado) MULTIPLICA linhas.
# Como codigo_ibge6 é único, a contagem não pode ter mudado:
assert len(sih) == linhas_antes_dos_joins, "FALHOU: o join alterou o nº de linhas!"
print(f"✅ Joins feitos sem alterar o nº de linhas ({len(sih):,})".replace(",", "."))

sih[["MUNIC_RES", "nome_mun_res", "uf_res", "MUNIC_MOV", "nome_mun_mov", "uf_mov"]].head()

✅ Joins feitos sem alterar o nº de linhas (258.125)


,MUNIC_RES,nome_mun_res,uf_res,MUNIC_MOV,nome_mun_mov,uf_mov
0,251080,Patos,PB,251080,Patos,PB
1,250670,Imaculada,PB,251080,Patos,PB
2,250939,Maturéia,PB,251080,Patos,PB
3,251210,Pombal,PB,250750,João Pessoa,PB
4,261160,Recife,PE,250750,João Pessoa,PB


### ✅ Validação 2 — 100% dos códigos com nome atribuído (zero órfãos)

"Órfão" = código de município presente no SIH que não encontrou correspondência na
tabela do IBGE (nome ficou vazio após o join). O critério de aceite da story exige
**zero órfãos** — ou, se existirem, que sejam listados e explicados um a um.

Nota: `MUNIC_MOV` (e também `MUNIC_RES`) pode conter municípios de fora da PB —
isso **não** é órfão nem erro, é informação valiosa (fluxo interestadual). Como a
tabela do IBGE é nacional, esses códigos também ganham nome normalmente.

In [8]:
orfaos_res = sih.loc[sih["nome_mun_res"].isna(), "MUNIC_RES"].value_counts()
orfaos_mov = sih.loc[sih["nome_mun_mov"].isna(), "MUNIC_MOV"].value_counts()

print(f"Códigos órfãos em MUNIC_RES (origem) : {len(orfaos_res)}")
print(f"Códigos órfãos em MUNIC_MOV (destino): {len(orfaos_mov)}")

if len(orfaos_res) > 0:
    print("\nÓrfãos de origem (código: nº de internações):")
    print(orfaos_res)
if len(orfaos_mov) > 0:
    print("\nÓrfãos de destino (código: nº de internações):")
    print(orfaos_mov)

assert len(orfaos_res) == 0 and len(orfaos_mov) == 0, (
    "FALHOU: existem códigos sem nome — listar e explicar antes de seguir!"
)
print("\n✅ VALIDAÇÃO 2 OK — 100% dos códigos de origem e destino têm nome")

Códigos órfãos em MUNIC_RES (origem) : 0
Códigos órfãos em MUNIC_MOV (destino): 0

✅ VALIDAÇÃO 2 OK — 100% dos códigos de origem e destino têm nome


### Espiada extra: de quais estados vêm os códigos?

Não é exigência da story, mas ajuda a entender a base que acabamos de montar (e
confirma na prática por que a tabela nacional foi a escolha certa).

In [9]:
print("Internações por UF de RESIDÊNCIA do paciente:")
print(sih["uf_res"].value_counts().to_string())
print("\nInternações por UF do hospital (destino):")
print(sih["uf_mov"].value_counts().to_string())

Internações por UF de RESIDÊNCIA do paciente:
uf_res
PB    256623
PE       760
RN       212
CE       121
SP        98
RJ        85
MA        30
AL        26
SC        24
BA        23
DF        18
MG        17
PR        16
GO        13
PA        12
AC         9
MT         8
PI         7
AM         5
SE         4
RR         3
RS         3
RO         2
TO         2
MS         2
ES         2

Internações por UF do hospital (destino):
uf_mov
PB    258125


## 7. Salvar o dataset tratado

Salvamos o resultado em `data/processed/` — a pasta de dados **derivados** na
arquitetura da disciplina (`data/raw/` guarda só o que veio da fonte, intocado).
Formato parquet: compacto, rápido de ler e preserva os tipos das colunas
(um CSV, por exemplo, perderia a distinção texto/número que ajustamos no passo 4).

Todo notebook e análise seguinte do projeto parte deste arquivo.

In [10]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
destino = DATA_PROCESSED / "sih_pb_2025_tratado.parquet"

sih.to_parquet(destino, index=False)

tamanho_mb = destino.stat().st_size / 1_000_000
print(f"Salvo: {destino}")
print(f"{len(sih):,} linhas x {sih.shape[1]} colunas · {tamanho_mb:.1f} MB".replace(",", "."))

Salvo: ..\data\processed\sih_pb_2025_tratado.parquet
258.125 linhas x 118 colunas · 11.5 MB


## 8. Registro final

Releitura do arquivo salvo para confirmar que o que está em disco é exatamente o que
validamos em memória (última fronteira de confiança antes de encerrar a story).

In [11]:
conferencia = pd.read_parquet(destino, columns=["MUNIC_RES", "nome_mun_res"])
assert len(conferencia) == soma_dos_meses, "FALHOU: arquivo salvo difere do validado!"
assert conferencia["nome_mun_res"].notna().all(), "FALHOU: nome vazio no arquivo salvo!"

print("RESUMO DA US-01")
print("=" * 50)
print(f"Arquivos mensais lidos        : {len(linhas_por_mes)}")
print(f"Soma das contagens dos meses  : {soma_dos_meses:,}".replace(",", "."))
print(f"Total do dataset unificado    : {total_unificado:,}".replace(",", "."))
print(f"Códigos órfãos (origem/destino): {len(orfaos_res)} / {len(orfaos_mov)}")
print(f"Arquivo final                 : {destino}")
print("=" * 50)
print("✅ US-01 concluída: base 2025 unificada, com nomes, validada e salva.")

RESUMO DA US-01
Arquivos mensais lidos        : 12
Soma das contagens dos meses  : 258.125
Total do dataset unificado    : 258.125
Códigos órfãos (origem/destino): 0 / 0
Arquivo final                 : ..\data\processed\sih_pb_2025_tratado.parquet
✅ US-01 concluída: base 2025 unificada, com nomes, validada e salva.
